# Region Comparison
Compare two spatial regions of interest (ROI) across tissues: DE + enrichment analysis.

In [ ]:
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import geopandas as gpd
from matplotlib.widgets import LassoSelector
from matplotlib.path import Path as MplPath
import ipympl
%matplotlib widget
import matplotlib
import spatialdata as spd
from spatialdata.transformations import get_transformation
from shapely.affinity import scale as shapely_scale
from shapely.geometry import Polygon
import scipy.sparse as sp
import gseapy as gp
from statsmodels.stats.multitest import multipletests


In [ ]:
import sys
_candidates = []
_nb = globals().get("__vsc_ipynb_file__")
if _nb:
    _candidates.append(Path(_nb).resolve().parent)
_candidates.append(Path.cwd().resolve())
_candidates.extend([
    Path("/home/janzules/spatial/CAR-T/code"),
    Path("/Users/janzules/Roselab/Spatial/CAR_T/code"),
])
_added = False
for _start in _candidates:
    for _parent in [_start, *_start.parents]:
        if (_parent / "misc" / "plot_style.py").exists():
            if str(_parent) not in sys.path:
                sys.path.insert(0, str(_parent))
            _added = True
            break
    if _added:
        break
if not _added:
    raise RuntimeError("Could not locate code/misc/plot_style.py")
from misc.plot_style import CELL_TYPE_COLORS, get_cell_colors


# Data Prep
## Locations
### Gemini

In [ ]:
zarr_file   = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400"
code_folder = Path("/home/janzules/spatial/CAR-T/code")
project_dir = Path("/coh_labs/yunroseli/Jona/CAR-T/")


### Laptop

In [ ]:
# project_dir = Path("/Users/janzules/Roselab/Spatial/CAR_T")
# zarr_file   = project_dir / "data/zarrFiles/CellCharterClusters"


## Loading Data

In [ ]:
sdata = spd.read_zarr(zarr_file)
adata = sdata.tables['segmentation_counts']


# Functions

In [ ]:
def read_gmt(gmt_path):
    gene_sets = {}
    rows = []
    with open(gmt_path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            gene_set    = parts[0]
            description = parts[1]
            genes       = parts[2:]
            gene_sets[gene_set] = genes
            rows.append({
                "gene_set":    gene_set,
                "description": description,
                "n_genes":     len(genes),
                "source_file": Path(gmt_path).name,
            })
    return gene_sets, pd.DataFrame(rows)


In [ ]:
pathway_dir = project_dir / "data/references/pathways"

gmt_files = {
    "hallmark": pathway_dir / "mh.all.v2026.1.Mm.symbols.gmt",
    "reactome": pathway_dir / "m2.cp.reactome.v2026.1.Mm.symbols.gmt",
    "go_bp":    pathway_dir / "m5.go.bp.v2026.1.Mm.symbols.gmt",
    "curated":  pathway_dir / "m2.cp.v2026.1.Mm.symbols.gmt",
}

all_gene_sets = {}
gene_set_metadata_list = []

for collection, path in gmt_files.items():
    gs, meta = read_gmt(path)
    meta["collection"] = collection
    all_gene_sets.update(gs)
    gene_set_metadata_list.append(meta)

gene_set_metadata = pd.concat(gene_set_metadata_list, ignore_index=True)
print(f"Total gene sets loaded: {len(all_gene_sets):,}")
display(gene_set_metadata.groupby("collection")[["gene_set"]].count())


In [ ]:
def get_tissue_transform(sdata, tissue_name):
    """Returns (scale, tx, ty) for shapes → image pixel mapping."""
    shapes_key = f"{tissue_name}_cell_boundaries"
    image_key  = f"{tissue_name}_hires_tissue_image"
    cs = "downscale_to_hires"
    shapes_tf   = get_transformation(sdata.shapes[shapes_key], get_all=True)[cs]
    scale       = float(shapes_tf.scale[0])
    image_tf    = get_transformation(sdata.images[image_key], get_all=True)[cs]
    translation = image_tf.transformations[0].translation
    ty, tx      = float(translation[1]), float(translation[2])
    return scale, tx, ty


In [ ]:
def plot_tissue_for_selection(sdata, adata, tissue_name,
                              pathway_1=None, top_quantile=0.25,
                              cell_type_col=None, cell_types_to_show=None,
                              highlight_types=None, highlight_buffer=25,
                              he_alpha=0.6, shrink=1, col_1="Blues"):
    """
    Render a tissue for interactive lasso selection.
    Returns (fig, ax, a_sub, coords_shared, scale) for use by the selector cell.
    """
    from matplotlib.patches import Patch

    cell_types_to_show = cell_types_to_show or []
    highlight_types    = highlight_types or []

    image_key    = f"{tissue_name}_hires_tissue_image"
    boundary_key = f"{tissue_name}_cell_boundaries"

    image  = sdata.images[image_key].values
    shapes = sdata.shapes[boundary_key].copy()

    shapes["geometry"] = shapes["geometry"].convex_hull.buffer(-shrink)
    scale, tx, ty = get_tissue_transform(sdata, tissue_name)
    shapes["geometry"] = shapes["geometry"].apply(
        lambda geom: shapely_scale(geom, xfact=scale, yfact=scale, origin=(0, 0))
    )

    mask  = adata.obs["tissue"] == tissue_name
    a_sub = adata[mask]

    join_cols = [c for c in [pathway_1, cell_type_col] if c is not None
                 and c in a_sub.obs.columns]
    shapes = shapes.join(a_sub.obs[join_cols], how="left")

    # Filtering
    if cell_type_col and cell_types_to_show:
        threshold    = shapes[pathway_1].quantile(1 - top_quantile) if pathway_1 else None
        shapes_other = shapes[
            (shapes[pathway_1] >= threshold) &
            (~shapes[cell_type_col].isin(cell_types_to_show))
        ].dropna(subset=[pathway_1]).copy() if pathway_1 else shapes[
            ~shapes[cell_type_col].isin(cell_types_to_show)].copy()
        shapes_ct_normal  = shapes[
            shapes[cell_type_col].isin(cell_types_to_show) &
            ~shapes[cell_type_col].isin(highlight_types)].copy()
        shapes_highlight  = shapes[shapes[cell_type_col].isin(highlight_types)].copy()
        shapes_highlight["geometry"] = shapes_highlight["geometry"].buffer(highlight_buffer)
    elif pathway_1:
        threshold  = shapes[pathway_1].quantile(1 - top_quantile)
        shapes_other = shapes[shapes[pathway_1] >= threshold].copy()
        shapes_ct_normal = shapes_highlight = None
    else:
        shapes_other = shapes.copy()
        shapes_ct_normal = shapes_highlight = None

    fig, ax = plt.subplots(figsize=(10, 10))
    img_rgb = np.transpose(image, (1, 2, 0))
    h, w    = img_rgb.shape[:2]
    ax.imshow(img_rgb, extent=[tx, tx + w, ty + h, ty], origin="upper", alpha=he_alpha)

    if pathway_1 and pathway_1 in shapes_other.columns:
        shapes_other.plot(column=pathway_1, cmap=col_1, ax=ax,
                          alpha=0.7, edgecolor="none", linewidth=0,
                          legend=True, legend_kwds={"label": pathway_1, "shrink": 0.4})
    else:
        shapes_other.plot(ax=ax, color="lightblue", edgecolor="none", linewidth=0, alpha=0.5)

    if shapes_ct_normal is not None and not shapes_ct_normal.empty:
        shapes_ct_normal.plot(ax=ax,
                              color=get_cell_colors(shapes_ct_normal[cell_type_col].astype(str)),
                              edgecolor="black", linewidth=0.4, alpha=0.9)
    if shapes_highlight is not None and not shapes_highlight.empty:
        shapes_highlight.plot(ax=ax,
                              color=get_cell_colors(shapes_highlight[cell_type_col].astype(str)),
                              edgecolor="white", linewidth=0.8, alpha=0.95)

    ax.set_title(f"{tissue_name} — draw lasso to select region", fontsize=12)
    ax.axis("off")
    plt.tight_layout()

    raw_coords    = adata.obsm["spatial"][mask.values]
    coords_shared = raw_coords * scale

    return fig, ax, a_sub, coords_shared, scale


# Pathway Scoring

In [ ]:
# Map short name → full MSigDB gene set name
# Search: gene_set_metadata[gene_set_metadata["gene_set"].str.contains("KEYWORD", case=False)]
pathway_panel = {
    "IFN_gamma":  "HALLMARK_INTERFERON_GAMMA_RESPONSE",
    "Glycolysis": "HALLMARK_GLYCOLYSIS",
    # add more here
}

# Validation
panel_check = []
for short_name, gene_set_name in pathway_panel.items():
    found = gene_set_name in all_gene_sets
    panel_check.append({
        "short_name":    short_name,
        "gene_set_name": gene_set_name,
        "found":         found,
        "n_genes":       len(all_gene_sets[gene_set_name]) if found else None,
        "collection":    gene_set_metadata.loc[
                             gene_set_metadata["gene_set"].eq(gene_set_name), "collection"
                         ].iloc[0] if found else "NOT FOUND",
    })
display(pd.DataFrame(panel_check))


In [ ]:
available_genes = set(adata.var_names.astype(str))
score_records   = []

for short_name, gene_set_name in pathway_panel.items():
    pathway_genes = all_gene_sets.get(gene_set_name, [])
    matched       = [g for g in pathway_genes if g in available_genes]
    score_col     = f"{short_name}_score"
    print(f"Scoring {short_name!r}  ({len(matched)}/{len(pathway_genes)} genes matched)...")
    if len(matched) < 5:
        print(f"  WARNING: too few matched genes — skipping.")
        continue
    sc.tl.score_genes(adata, gene_list=matched, score_name=score_col, use_raw=False)
    score_records.append({"short_name": short_name, "score_col": score_col, "n_matched": len(matched)})

print("\nDone. Score columns:")
display(pd.DataFrame(score_records))


# Region Selection
## Region 1

In [ ]:
# --- Inputs for Region 1 ---
tissue_1       = "RTCyPSCA_2_3"
pathway_1      = "IFN_gamma_score"
top_quantile   = 0.25
cell_type_col  = None
cell_types_to_show = []
he_alpha       = 0.6
shrink         = 1

fig1, ax1, a_sub1, coords1, scale1 = plot_tissue_for_selection(
    sdata, adata, tissue_1,
    pathway_1=pathway_1,
    top_quantile=top_quantile,
    cell_type_col=cell_type_col,
    cell_types_to_show=cell_types_to_show,
    he_alpha=he_alpha,
    shrink=shrink,
)
plt.show()


In [ ]:
# --- Lasso selector for Region 1 ---
# Draw the lasso on the figure above, then run the extraction cell below.
state1 = {"mask": None}

def onselect1(verts):
    path  = MplPath(verts)
    inside = path.contains_points(coords1)
    state1["mask"] = inside
    n = inside.sum()
    ax1.set_title(f"Region 1 ({tissue_1}): {n} cells selected", fontsize=12)
    fig1.canvas.draw_idle()
    print(f"Region 1: {n} cells selected")

lasso1 = LassoSelector(ax1, onselect1, button=[1])
print("Draw lasso on the figure above. Selection stores in state1.")


In [ ]:
assert state1["mask"] is not None, "Draw a lasso on Region 1 first"

adata_roi1 = a_sub1[state1["mask"]].copy()
adata_roi1.obs["roi"] = "Region_1"
print(f"Region 1: {adata_roi1.n_obs} cells from {tissue_1}")
print(adata_roi1.obs["c2l_permissive"].value_counts())


## Region 2

In [ ]:
# --- Inputs for Region 2 ---
tissue_2       = "RTCyPSCA_1_3"
pathway_2_vis  = "IFN_gamma_score"
top_quantile_2 = 0.25

fig2, ax2, a_sub2, coords2, scale2 = plot_tissue_for_selection(
    sdata, adata, tissue_2,
    pathway_1=pathway_2_vis,
    top_quantile=top_quantile_2,
    he_alpha=0.6,
    shrink=1,
)
plt.show()


In [ ]:
# --- Lasso selector for Region 2 ---
state2 = {"mask": None}

def onselect2(verts):
    path   = MplPath(verts)
    inside = path.contains_points(coords2)
    state2["mask"] = inside
    n = inside.sum()
    ax2.set_title(f"Region 2 ({tissue_2}): {n} cells selected", fontsize=12)
    fig2.canvas.draw_idle()
    print(f"Region 2: {n} cells selected")

lasso2 = LassoSelector(ax2, onselect2, button=[1])
print("Draw lasso on the figure above. Selection stores in state2.")


In [ ]:
assert state2["mask"] is not None, "Draw a lasso on Region 2 first"

adata_roi2 = a_sub2[state2["mask"]].copy()
adata_roi2.obs["roi"] = "Region_2"
print(f"Region 2: {adata_roi2.n_obs} cells from {tissue_2}")
print(adata_roi2.obs["c2l_permissive"].value_counts())


# Differential Expression: Region 1 vs Region 2

In [ ]:
import anndata

# Combine both ROIs — align on shared genes only
shared_genes = adata_roi1.var_names.intersection(adata_roi2.var_names)
adata_combined = anndata.concat(
    [adata_roi1[:, shared_genes], adata_roi2[:, shared_genes]],
    label="roi", keys=["Region_1", "Region_2"]
)
adata_combined.obs["roi"] = adata_combined.obs["roi"].astype("category")
print(f"Combined: {adata_combined.n_obs} cells, {adata_combined.n_vars} genes")
print(adata_combined.obs["roi"].value_counts())


In [ ]:
# DE: Region 1 vs Region 2 (all cell types pooled)
sc.tl.rank_genes_groups(
    adata_combined,
    groupby="roi",
    groups=["Region_1"],
    reference="Region_2",
    method="wilcoxon",
    key_added="de_roi",
)

de_df = sc.get.rank_genes_groups_df(adata_combined, group="Region_1", key="de_roi")
de_df = de_df.sort_values("scores", ascending=False)

print(f"Total DE genes: {len(de_df)}")
print("\nTop upregulated in Region 1:")
display(de_df.query("pvals_adj < 0.05 and logfoldchanges > 0.5").head(20))
print("\nTop downregulated in Region 1:")
display(de_df.query("pvals_adj < 0.05 and logfoldchanges < -0.5").tail(20))


In [ ]:
# Optional: DE per cell type (only cell types present in both ROIs)
cell_type_col = "c2l_permissive"

ct_roi1 = set(adata_roi1.obs[cell_type_col].unique())
ct_roi2 = set(adata_roi2.obs[cell_type_col].unique())
shared_cts = sorted(ct_roi1 & ct_roi2)
print(f"Cell types present in both ROIs: {shared_cts}")

de_per_ct = {}
for ct in shared_cts:
    sub = adata_combined[adata_combined.obs[cell_type_col] == ct].copy()
    if sub.obs["roi"].nunique() < 2:
        continue
    n1 = (sub.obs["roi"] == "Region_1").sum()
    n2 = (sub.obs["roi"] == "Region_2").sum()
    if n1 < 3 or n2 < 3:
        print(f"Skipping {ct} — too few cells (R1={n1}, R2={n2})")
        continue
    sc.tl.rank_genes_groups(sub, groupby="roi", groups=["Region_1"],
                             reference="Region_2", method="wilcoxon",
                             key_added="de_ct")
    df = sc.get.rank_genes_groups_df(sub, group="Region_1", key="de_ct")
    de_per_ct[ct] = df.sort_values("scores", ascending=False)
    print(f"{ct}: {(df.pvals_adj < 0.05).sum()} significant DE genes")


# Enrichment Analysis

In [ ]:
# Enrichment on pooled DE (Region 1 upregulated genes)
databases = [
    "KEGG_2019_Mouse",
    "GO_Biological_Process_2023",
    "GO_Molecular_Function_2023",
]

up_genes = (
    de_df
    .query("pvals_adj < 0.05 and logfoldchanges > 0.5")
    .head(300)["names"]
    .tolist()
)
down_genes = (
    de_df
    .query("pvals_adj < 0.05 and logfoldchanges < -0.5")
    .tail(300)["names"]
    .tolist()
)

print(f"Upregulated genes for enrichment: {len(up_genes)}")
print(f"Downregulated genes for enrichment: {len(down_genes)}")

enr_up   = gp.enrichr(gene_list=up_genes,   gene_sets=databases, organism="mouse", outdir=None)
enr_down = gp.enrichr(gene_list=down_genes, gene_sets=databases, organism="mouse", outdir=None)


In [ ]:
# Summary of enrichment results
for label, enr in [("Region_1 UP", enr_up), ("Region_1 DOWN", enr_down)]:
    sig = enr.results[enr.results["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
    print(f"\n=== {label} ({len(sig)} significant terms) ===")
    print(sig[["Gene_set", "Term", "Overlap", "Adjusted P-value"]].head(15).to_string(index=False))


In [ ]:
# Enrichment per cell type (uses de_per_ct from DE cell above)
enr_per_ct = {}

for ct, df in de_per_ct.items():
    markers = df.query("pvals_adj < 0.05 and logfoldchanges > 0.5").head(200)["names"].tolist()
    if len(markers) < 10:
        print(f"Skipping {ct} — too few markers")
        continue
    print(f"Running enrichment for {ct} ({len(markers)} markers)...")
    enr_per_ct[ct] = gp.enrichr(
        gene_list=markers, gene_sets=databases, organism="mouse", outdir=None
    ).results

# Summary
for ct, df in enr_per_ct.items():
    sig = df[df["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
    print(f"\n=== {ct} ({len(sig)} significant terms) ===")
    print(sig[["Gene_set", "Term", "Overlap", "Adjusted P-value"]].head(10).to_string(index=False))


# Save Results

In [ ]:
out_dir = project_dir / "results/region_comparison"
out_dir.mkdir(parents=True, exist_ok=True)

# DE results
de_df.to_csv(out_dir / f"{tissue_1}_vs_{tissue_2}_DE_pooled.csv", index=False)
for ct, df in de_per_ct.items():
    df.to_csv(out_dir / f"{tissue_1}_vs_{tissue_2}_DE_{ct}.csv", index=False)

# Enrichment
enr_up.results.to_csv(out_dir / f"{tissue_1}_vs_{tissue_2}_enrichr_up.csv", index=False)
enr_down.results.to_csv(out_dir / f"{tissue_1}_vs_{tissue_2}_enrichr_down.csv", index=False)
for ct, df in enr_per_ct.items():
    df.to_csv(out_dir / f"{tissue_1}_vs_{tissue_2}_enrichr_{ct}.csv", index=False)

print(f"Saved all results to {out_dir}")
